In [136]:
## analyze data saved with the lab jack
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import scipy.signal as sp

In [ ]:
data_dir = "../data/20250317/"

## Files for 1-50 Hz noise drive
curr_files = glob(data_dir + "transfer_func_data_noise_0.100Hz_20.000V*.npz")
ref_files = glob(data_dir + "transfer_func_data_noise_0.100Hz_0.002V*.npz")

nperseg = 2**14

channel_list = ['x', 'y', 'Sum', 'Drive']

print("Found %d current files and %d reference files" % (len(curr_files), len(ref_files)))

Found 0 current files and 0 reference files


In [138]:
def make_psd_from_file_list(files):
    psd_dict = {}
    n_psds = 0

    for ch in channel_list:
        psd_dict[ch] = None

    for f in files:
        data = np.load(f)
    
        for meas in range(len(data['data'])):
            curr_data = data['data'][meas]
            n_psds += 1

            for ch_idx, ch in enumerate(channel_list):
                freqs, cpsd = sp.welch(curr_data[:,ch_idx], nperseg=nperseg, fs=data['Fs'])

                if(psd_dict[ch] is None):
                    psd_dict[ch] = cpsd
                else:
                    psd_dict[ch] += cpsd

    for ch in channel_list:
        psd_dict[ch] /= n_psds

    return psd_dict, freqs

In [139]:
def plot_psd_dict(psd_dict, freqs, fig):
    plt.figure(fig.number)
    for ch in channel_list:
        plt.subplot(1, len(channel_list), channel_list.index(ch)+1)
        plt.title(ch)
        plt.semilogy(freqs, psd_dict[ch])
        plt.xlim(0,55)
        plt.xlabel("Frequency (Hz)")

    plt.subplot(1, len(channel_list), 1)        
    plt.ylabel("PSD [V$^2$/Hz]")
    

In [140]:
curr_psd_dict, curr_freqs = make_psd_from_file_list(curr_files)
ref_psd_dict, ref_freqs = make_psd_from_file_list(ref_files)

fig = plt.figure(figsize=(15,3))
plot_psd_dict(curr_psd_dict, curr_freqs, fig)
plot_psd_dict(ref_psd_dict, ref_freqs, fig)

plt.show()

TypeError: unsupported operand type(s) for /=: 'NoneType' and 'int'